# Lecture 7: Information Extraction — Answer Key
### NLP Course 2027 — Week 04

---
Complete answers for all four exercises in Lecture 7.

In [ ]:
import re, nltk
import numpy as np
from nltk import word_tokenize, pos_tag, ne_chunk, sent_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict

for pkg in ['maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words',
            'stopwords', 'punkt', 'averaged_perceptron_tagger']:
    nltk.download(pkg, quiet=True)
print('Ready')

---
## Exercise 7.1 — WORKS_FOR Relation Extraction

**Task:** Extract (person, organization) WORKS_FOR relations using regex patterns.

**Key concept:** Pattern-based relation extraction is high-precision but low-recall. Each surface form ('works at', 'joined', 'employed by') needs its own pattern.

In [ ]:
def extract_works_for(text):
    """Extract WORKS_FOR relations: (person, organization)."""
    patterns = [
        r'([A-Z][a-z]+ [A-Z][a-z]+) works? (?:at|for) ([A-Z][\w\s]+?)(?:[.,]|$)',
        r'([A-Z][a-z]+ [A-Z][a-z]+) is employed by ([A-Z][\w\s]+?)(?:[.,]|$)',
        r'([A-Z][a-z]+ [A-Z][a-z]+) joined ([A-Z][\w\s]+?) ',
        r'([A-Z][a-z]+ [A-Z][a-z]+) is a \w+ at ([A-Z][\w\s]+?)(?:[.,]|$)',
        r'([A-Z][a-z]+ [A-Z][a-z]+),? (?:CEO|CTO|CFO|director|researcher|engineer) (?:of|at) ([A-Z][\w\s]+?)(?:[.,]|$)',
    ]
    results = []
    for pat in patterns:
        for m in re.finditer(pat, text):
            results.append(('WORKS_FOR', m.group(1).strip(), m.group(2).strip()))
    return results

test_text = """
John Smith works at Google. Jane Doe is employed by Microsoft.
Bob Johnson joined Apple last year. Sarah Lee is a researcher at MIT.
Elon Musk, CEO of Tesla, announced the new model.
"""
for rel in extract_works_for(test_text):
    print(rel)

---
## Exercise 7.2 — RAKE vs TF-IDF Keyphrase Extraction

**Task:** Compare both methods on a news article.

**Key concept:** RAKE scores by word co-occurrence within candidate phrases (ratio of degree to frequency). TF-IDF scores individual words by document-level rarity. RAKE tends to produce multi-word phrases; TF-IDF single words.

In [ ]:
def rake_keywords(text, top_n=10):
    """RAKE: split on stopwords/punctuation, score by word degree/frequency ratio."""
    stop = set(stopwords.words('english')) | set('.,;:!?()[]{}"\'')
    # Split text into candidate phrases
    pattern = r'[^\w\s]|\b(?:' + '|'.join(re.escape(w) for w in stop) + r')\b'
    phrases = [p.strip() for p in re.split(pattern, text.lower()) if p.strip()]
    phrases = [p for p in phrases if len(p.split()) >= 1]

    word_freq   = defaultdict(int)
    word_degree = defaultdict(int)
    for phrase in phrases:
        words = phrase.split()
        for w in words:
            word_freq[w]   += 1
            word_degree[w] += len(words) - 1  # co-occurrence with other words in phrase

    word_score = {w: (word_degree[w] + word_freq[w]) / word_freq[w]
                  for w in word_freq}

    phrase_scores = []
    for phrase in set(phrases):
        score = sum(word_score.get(w, 0) for w in phrase.split())
        phrase_scores.append((phrase, score))

    return sorted(phrase_scores, key=lambda x: -x[1])[:top_n]


def tfidf_keyphrases(text, top_n=10):
    """TF-IDF: treat each sentence as a document, score words."""
    sentences = sent_tokenize(text)
    if len(sentences) < 2:
        sentences = [text]
    vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    tfidf_matrix = vec.fit_transform(sentences)
    scores = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
    terms  = vec.get_feature_names_out()
    ranked = sorted(zip(terms, scores), key=lambda x: -x[1])
    return ranked[:top_n]


article = """
Artificial intelligence researchers at Stanford University have developed a new
deep learning model that can diagnose rare diseases from medical images with
95% accuracy. The model, trained on over one million patient records, outperforms
human specialists in identifying specific genetic disorders. The research team,
led by Professor Sarah Chen, plans to conduct clinical trials next year.
"""

print('RAKE keyphrases:')
for phrase, score in rake_keywords(article):
    print(f'  {score:.2f}  {phrase}')

print('\nTF-IDF keyphrases:')
for phrase, score in tfidf_keyphrases(article):
    print(f'  {score:.4f}  {phrase}')

---
## Exercise 7.3 — Event Extraction

**Task:** Extract {who, action, when, where} from news sentences.

**Key concept:** Event extraction combines NER (who/where) with regex (when) and POS patterns (action/verb). Lightweight regex works for well-structured news; deep learning is needed for complex events.

In [ ]:
def extract_events(text):
    """Extract events as {who, action, when, where} dicts."""
    events = []
    when_pattern  = r'(?:on\s+)?(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday|last\s+\w+|next\s+\w+|yesterday|today|\w+day|(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d+)'
    where_pattern = r'in ([A-Z][a-zA-Z\s]+?)(?:\s+on|\s+last|\s*[.,])'

    for sent in sent_tokenize(text):
        event = {'who': None, 'action': None, 'when': None, 'where': None}

        # Extract named entities for WHO and WHERE
        ne_tree = ne_chunk(pos_tag(word_tokenize(sent)))
        persons, places = [], []
        for subtree in ne_tree:
            if hasattr(subtree, 'label'):
                name = ' '.join(w for w, _ in subtree.leaves())
                if subtree.label() == 'PERSON': persons.append(name)
                if subtree.label() in ('GPE', 'LOCATION'): places.append(name)

        event['who']   = persons[0] if persons else None
        event['where'] = places[0]  if places  else None

        # Extract WHEN with regex
        m = re.search(when_pattern, sent, re.IGNORECASE)
        if m: event['when'] = m.group().strip()

        # Extract ACTION: first verb phrase after subject
        verbs = [w for w, t in pos_tag(word_tokenize(sent)) if t.startswith('VB')]
        event['action'] = verbs[0] if verbs else None

        if any(v for v in event.values()):
            events.append(event)
    return events

news = """
Google announced a new AI model in San Francisco on Monday.
Tesla CEO Elon Musk launched the Cybertruck in Texas last November.
The World Health Organization declared a public health emergency in Geneva on Tuesday.
"""
for event in extract_events(news):
    print(event)

---
## Exercise 7.4 — Simple Coreference Resolution

**Task:** Replace pronouns (he/she/they) with the most recent matching named entity.

**Key concept:** Naive coreference uses recency: replace a pronoun with the last seen PERSON entity. This fails for complex cases (multiple people, ambiguous pronouns) — full coreference needs neural models like SpanBERT.

In [ ]:
def simple_coref_resolve(text):
    """Replace he/she/they with the most recent PERSON entity."""
    sentences = sent_tokenize(text)
    last_person = None
    resolved_sents = []

    for sent in sentences:
        # Find named entities first
        ne_tree = ne_chunk(pos_tag(word_tokenize(sent)))
        for subtree in ne_tree:
            if hasattr(subtree, 'label') and subtree.label() == 'PERSON':
                last_person = ' '.join(w for w, _ in subtree.leaves())

        # Replace pronouns
        if last_person:
            sent = re.sub(r'\bHe\b', last_person, sent)
            sent = re.sub(r'\bShe\b', last_person, sent)
            sent = re.sub(r'\bhe\b', last_person.lower().split()[0], sent)
            sent = re.sub(r'\bshe\b', last_person.lower().split()[0], sent)
        resolved_sents.append(sent)

    return ' '.join(resolved_sents)

text = """Barack Obama was born in Hawaii. He served as president for eight years.
Michelle Obama is a lawyer. She wrote a bestselling memoir."""

print('Original:')
print(text)
print('\nResolved:')
print(simple_coref_resolve(text))
print()
print('Limitation: This approach fails when multiple people are mentioned in the same sentence.')

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 7.1 | Pattern-based RE: high precision, low recall; one pattern per surface form |
| 7.2 | RAKE produces phrases; TF-IDF produces words — both complementary |
| 7.3 | Event extraction = NER (who/where) + regex (when) + POS (action) |
| 7.4 | Naive coreference uses recency heuristic; neural models needed for robustness |

---
*NLP Course 2027 — Week 04*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**